# NDVI in 2013 Forest Classes Before vs After Hurricane Melissa

This notebook replicates the NDVI before/after workflow for **forests** (not mangroves), using:
- 2013 landcover dataset (`2013_landuse_landcover.gpkg`)
- Class definitions in `Robyn_catchment_analysis.py`:
  - `forest_flood_equivalent_classes`
  - `mixed_land_use_fractions` (forest fraction for mixed classes)

Important:
- This notebook is separate and does **not** modify your mangrove notebook or the `.py` source.
- Mixed classes are handled with fractional forest weights.


In [ ]:
from pathlib import Path
import ast

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches

plt.style.use('default')
pd.set_option('display.max_columns', 120)


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'dphil_papers').exists():
            return p
    raise FileNotFoundError(f'Could not find project root from {start}')

ROOT = find_project_root(Path.cwd())

ndvi_dir = ROOT / 'dphil_papers/dphil_paper_3/inputs/ndvi'

def pick_ndvi(pattern: str) -> Path:
    matches = sorted(ndvi_dir.glob(pattern))
    if len(matches) == 0:
        raise FileNotFoundError(f'No NDVI file matched pattern: {pattern} in {ndvi_dir}')
    if len(matches) > 1:
        names = '\n'.join(str(m.name) for m in matches)
        raise FileExistsError(f'Ambiguous NDVI pattern: {pattern} matched multiple files:\n{names}')
    return matches[0]

ndvi_before_path = pick_ndvi('HLS_masked_NDVI_2months_before_epsg3448*.tif')
ndvi_after_path = pick_ndvi('HLS_masked_NDVI_2months_after*_epsg3448*.tif')
landcover_path = ROOT / 'dphil_papers/dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_landcover.gpkg'
robyn_defs_path = ROOT / 'dphil_papers/robyns_libraries/Robyn_catchment_analysis.py'

output_dir = ROOT / 'dphil_papers/dphil_paper_3/results/threats/ndvi/draft_processed_images'
output_dir.mkdir(parents=True, exist_ok=True)

# Control exports while iterating
SAVE_PNGS = False
SAVE_CSVS = True

print('Working dir:', Path.cwd())
print('Project root:', ROOT)
for p in [ndvi_before_path, ndvi_after_path, landcover_path, robyn_defs_path]:
    print(p.name, 'exists ->', p.exists())
print('Output dir:', output_dir)


In [ ]:
# Parse class definitions from Robyn_catchment_analysis.py without modifying it.
source_text = robyn_defs_path.read_text(encoding='utf-8')
module = ast.parse(source_text)

forest_flood_equivalent_classes = None
mixed_land_use_fractions_primary = None

for node in module.body:
    if not isinstance(node, ast.Assign):
        continue
    for target in node.targets:
        if not isinstance(target, ast.Name):
            continue

        if target.id == 'forest_flood_equivalent_classes' and forest_flood_equivalent_classes is None:
            forest_flood_equivalent_classes = set(ast.literal_eval(node.value))

        if target.id == 'mixed_land_use_fractions':
            candidate = ast.literal_eval(node.value)
            if (
                isinstance(candidate, dict)
                and any(isinstance(v, dict) and 'forest_flood_equivalent_classes' in v for v in candidate.values())
            ):
                mixed_land_use_fractions_primary = candidate

if forest_flood_equivalent_classes is None or mixed_land_use_fractions_primary is None:
    raise ValueError('Could not parse required class definitions from Robyn_catchment_analysis.py')

print('Forest flood-equivalent classes:', len(forest_flood_equivalent_classes))
print('Mixed classes with fractions:', len(mixed_land_use_fractions_primary))
print('Example mixed definitions:')
for k, v in list(mixed_land_use_fractions_primary.items())[:3]:
    print(' ', k, '->', v)


In [ ]:
with rasterio.open(ndvi_before_path) as src_b, rasterio.open(ndvi_after_path) as src_a:
    before_meta = {
        'crs': str(src_b.crs),
        'shape': (src_b.height, src_b.width),
        'count': src_b.count,
        'dtype': src_b.dtypes[0],
        'nodata': src_b.nodata,
        'bounds': src_b.bounds,
        'transform': src_b.transform,
    }
    after_meta = {
        'crs': str(src_a.crs),
        'shape': (src_a.height, src_a.width),
        'count': src_a.count,
        'dtype': src_a.dtypes[0],
        'nodata': src_a.nodata,
        'bounds': src_a.bounds,
        'transform': src_a.transform,
    }

same_grid = (
    before_meta['crs'] == after_meta['crs']
    and before_meta['shape'] == after_meta['shape']
    and before_meta['transform'] == after_meta['transform']
)

print('Before metadata:')
print(pd.Series({k: v for k, v in before_meta.items() if k != 'transform'}))
print('\nAfter metadata:')
print(pd.Series({k: v for k, v in after_meta.items() if k != 'transform'}))
print('\nSame grid:', same_grid)

if not same_grid:
    raise ValueError('Before/after NDVI rasters are not on same grid; resample first.')


In [ ]:
def forest_fraction_for_class(class_name: str) -> float:
    # Mixed classes: use explicit forest fraction from mixed_land_use_fractions
    if class_name in mixed_land_use_fractions_primary:
        return float(mixed_land_use_fractions_primary[class_name].get('forest_flood_equivalent_classes', 0.0))
    # Pure forest-equivalent classes
    if class_name in forest_flood_equivalent_classes:
        return 1.0
    return 0.0

landcover = gpd.read_file(landcover_path, columns=['Classify', 'geometry'])
with rasterio.open(ndvi_before_path) as src:
    raster_crs = src.crs
    raster_transform = src.transform
    raster_shape = (src.height, src.width)

if landcover.crs != raster_crs:
    landcover = landcover.to_crs(raster_crs)

landcover = landcover[landcover.geometry.notnull() & ~landcover.geometry.is_empty].copy()
landcover['forest_fraction'] = landcover['Classify'].astype(str).map(forest_fraction_for_class)
forest_landcover = landcover[landcover['forest_fraction'] > 0].copy()

print('Landcover polygons total:', f'{len(landcover):,}')
print('Forest-relevant polygons:', f'{len(forest_landcover):,}')
print('Unique forest-relevant classes:', forest_landcover['Classify'].nunique())
print('Forest-equivalent area in polygons (ha):', round((forest_landcover.geometry.area * forest_landcover['forest_fraction']).sum() / 10_000, 2))

# Rasterize class fractions to NDVI grid (0..1 forest-equivalent weight per pixel)
shapes = ((geom, float(frac)) for geom, frac in zip(forest_landcover.geometry, forest_landcover['forest_fraction']))
forest_fraction_raster = rasterize(
    shapes=shapes,
    out_shape=raster_shape,
    transform=raster_transform,
    fill=0.0,
    dtype='float32'
)

print('Rasterized forest-equivalent pixel sum:', float(forest_fraction_raster.sum()))


In [ ]:
with rasterio.open(ndvi_before_path) as src_b, rasterio.open(ndvi_after_path) as src_a:
    ndvi_before = src_b.read(1)
    ndvi_after = src_a.read(1)
    bounds = src_b.bounds

    valid_before = np.isfinite(ndvi_before) & (ndvi_before >= -1.0) & (ndvi_before <= 1.0)
    valid_after = np.isfinite(ndvi_after) & (ndvi_after >= -1.0) & (ndvi_after <= 1.0)

    if src_b.nodata is not None and np.isfinite(src_b.nodata):
        valid_before &= ndvi_before != src_b.nodata
    if src_a.nodata is not None and np.isfinite(src_a.nodata):
        valid_after &= ndvi_after != src_a.nodata

valid_forest = forest_fraction_raster > 0
valid_before_forest = valid_before & valid_forest
valid_after_forest = valid_after & valid_forest
valid_paired_forest = valid_before & valid_after & valid_forest

ndvi_before_forest = ndvi_before[valid_before_forest]
ndvi_after_forest = ndvi_after[valid_after_forest]
ndvi_before_paired = ndvi_before[valid_paired_forest]
ndvi_after_paired = ndvi_after[valid_paired_forest]
ndvi_delta_paired = ndvi_after_paired - ndvi_before_paired
weights_paired = forest_fraction_raster[valid_paired_forest].astype('float64')

print('Forest pixels (before-valid):', f'{ndvi_before_forest.size:,}')
print('Forest pixels (after-valid):', f'{ndvi_after_forest.size:,}')
print('Forest pixels (paired valid):', f'{ndvi_delta_paired.size:,}')
print('Weighted forest-equivalent paired pixel sum:', f'{weights_paired.sum():,.2f}')


In [ ]:
def weighted_mean(values, weights):
    return float(np.average(values, weights=weights))

summary_df = pd.DataFrame([
    {
        'series': 'before (paired forest pixels)',
        'n_pixels': int(ndvi_before_paired.size),
        'weighted_forest_equiv_pixels': float(weights_paired.sum()),
        'mean_unweighted': float(np.mean(ndvi_before_paired)),
        'mean_weighted': weighted_mean(ndvi_before_paired, weights_paired),
        'median_unweighted': float(np.median(ndvi_before_paired)),
        'p25_unweighted': float(np.percentile(ndvi_before_paired, 25)),
        'p75_unweighted': float(np.percentile(ndvi_before_paired, 75)),
    },
    {
        'series': 'after (paired forest pixels)',
        'n_pixels': int(ndvi_after_paired.size),
        'weighted_forest_equiv_pixels': float(weights_paired.sum()),
        'mean_unweighted': float(np.mean(ndvi_after_paired)),
        'mean_weighted': weighted_mean(ndvi_after_paired, weights_paired),
        'median_unweighted': float(np.median(ndvi_after_paired)),
        'p25_unweighted': float(np.percentile(ndvi_after_paired, 25)),
        'p75_unweighted': float(np.percentile(ndvi_after_paired, 75)),
    },
    {
        'series': 'delta after-before (paired forest pixels)',
        'n_pixels': int(ndvi_delta_paired.size),
        'weighted_forest_equiv_pixels': float(weights_paired.sum()),
        'mean_unweighted': float(np.mean(ndvi_delta_paired)),
        'mean_weighted': weighted_mean(ndvi_delta_paired, weights_paired),
        'median_unweighted': float(np.median(ndvi_delta_paired)),
        'p25_unweighted': float(np.percentile(ndvi_delta_paired, 25)),
        'p75_unweighted': float(np.percentile(ndvi_delta_paired, 75)),
    },
]).set_index('series').round(4)

summary_df


In [ ]:
# Mutually exclusive change categories (small changes count as up/down)
impact_indicators = {
    'n_paired_pixels': int(ndvi_delta_paired.size),
    'weighted_forest_equiv_pixels': float(weights_paired.sum()),
    'mean_delta_ndvi_unweighted': float(np.mean(ndvi_delta_paired)),
    'mean_delta_ndvi_weighted': float(np.average(ndvi_delta_paired, weights=weights_paired)),
    'median_delta_ndvi_unweighted': float(np.median(ndvi_delta_paired)),
    'pct_down_delta_lt_0_unweighted': float(100.0 * np.mean(ndvi_delta_paired < 0.0)),
    'pct_same_delta_eq_0_unweighted': float(100.0 * np.mean(ndvi_delta_paired == 0.0)),
    'pct_up_delta_gt_0_unweighted': float(100.0 * np.mean(ndvi_delta_paired > 0.0)),
    'pct_down_delta_lt_0_weighted': float(100.0 * weights_paired[ndvi_delta_paired < 0.0].sum() / weights_paired.sum()),
    'pct_same_delta_eq_0_weighted': float(100.0 * weights_paired[ndvi_delta_paired == 0.0].sum() / weights_paired.sum()),
    'pct_up_delta_gt_0_weighted': float(100.0 * weights_paired[ndvi_delta_paired > 0.0].sum() / weights_paired.sum()),
}

impact_df = pd.DataFrame([impact_indicators]).T.rename(columns={0: 'value'})
impact_df.index.name = 'indicator'
impact_df['value'] = impact_df['value'].astype(float)
impact_df


In [ ]:
# Weighted histogram: NDVI before vs after in forest-equivalent pixels
fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

low = min(np.percentile(ndvi_before_paired, 0.5), np.percentile(ndvi_after_paired, 0.5))
high = max(np.percentile(ndvi_before_paired, 99.5), np.percentile(ndvi_after_paired, 99.5))
bins = np.linspace(low, high, 90)

ax.hist(ndvi_before_paired, bins=bins, weights=weights_paired, alpha=0.50, density=True, label='Before event')
ax.hist(ndvi_after_paired, bins=bins, weights=weights_paired, alpha=0.50, density=True, label='After event')
ax.set_xlabel('NDVI')
ax.set_ylabel('Weighted density')
ax.set_title('Forest-equivalent NDVI Distribution: Before vs After')
ax.legend(loc='upper left')

before_after_hist_png = output_dir / 'ndvi_forests_before_after_hist_weighted.png'
if SAVE_PNGS:
    fig.savefig(before_after_hist_png, dpi=300)
    print('Saved:', before_after_hist_png)
else:
    print('PNG export skipped (SAVE_PNGS=False):', before_after_hist_png)
plt.show()


In [ ]:
# Weighted histogram: NDVI delta in forest-equivalent paired pixels
fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

q_low, q_high = np.percentile(ndvi_delta_paired, [0.5, 99.5])
bins = np.linspace(q_low, q_high, 90)

ax.hist(ndvi_delta_paired, bins=bins, weights=weights_paired, alpha=0.75, color='#3366cc', density=True)
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('NDVI change (after - before)')
ax.set_ylabel('Weighted density')
ax.set_title('Forest-equivalent NDVI Change Distribution (Paired Pixels)')

delta_hist_png = output_dir / 'ndvi_forests_change_hist_weighted.png'
if SAVE_PNGS:
    fig.savefig(delta_hist_png, dpi=300)
    print('Saved:', delta_hist_png)
else:
    print('PNG export skipped (SAVE_PNGS=False):', delta_hist_png)
plt.show()


In [ ]:
# Geospatial direction map in forest areas: down/same/up (strict sign)
cls = np.full(ndvi_before.shape, np.nan, dtype='float32')
cls[valid_paired_forest & ((ndvi_after - ndvi_before) < 0.0)] = -1.0
cls[valid_paired_forest & ((ndvi_after - ndvi_before) == 0.0)] = 0.0
cls[valid_paired_forest & ((ndvi_after - ndvi_before) > 0.0)] = 1.0

n_down = int(np.nansum(cls == -1.0))
n_same = int(np.nansum(cls == 0.0))
n_up = int(np.nansum(cls == 1.0))
n_total = n_down + n_same + n_up

cmap = ListedColormap(['#d73027', '#bdbdbd', '#1a9850'])
cmap.set_bad(color='white', alpha=1.0)
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)

fig, ax = plt.subplots(figsize=(12.5, 9.5), constrained_layout=True)
ax.imshow(
    cls,
    cmap=cmap,
    norm=norm,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    origin='upper',
    interpolation='nearest',
)
ax.set_title('Forest-equivalent NDVI Change Direction (After - Before)\nSame = exact zero change', fontsize=12)
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

legend_handles = [
    mpatches.Patch(facecolor='#d73027', edgecolor='none', label='NDVI down'),
    mpatches.Patch(facecolor='#bdbdbd', edgecolor='none', label='Same'),
    mpatches.Patch(facecolor='#1a9850', edgecolor='none', label='NDVI up'),
]
ax.legend(
    handles=legend_handles,
    loc='upper right',
    frameon=True,
    framealpha=0.92,
    fontsize=9,
    handlelength=0.9,
    handleheight=0.9,
    borderpad=0.35,
    labelspacing=0.3,
)

ax.text(
    0.01,
    0.01,
    f'n={n_total:,}  down={100*n_down/n_total:.1f}%  same={100*n_same/n_total:.1f}%  up={100*n_up/n_total:.1f}%',
    transform=ax.transAxes,
    fontsize=9,
    ha='left',
    va='bottom',
    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85, edgecolor='none')
)

map_png = output_dir / 'ndvi_forests_change_direction_map_strict_sign.png'
if SAVE_PNGS:
    fig.savefig(map_png, dpi=320)
    print('Saved:', map_png)
else:
    print('PNG export skipped (SAVE_PNGS=False):', map_png)
plt.show()


In [ ]:
# Optional exports
if SAVE_CSVS:
    impact_csv = output_dir / 'ndvi_forests_impact_indicators.csv'
    impact_df.reset_index().to_csv(impact_csv, index=False)
    print('Saved:', impact_csv)

    paired_csv = output_dir / 'ndvi_forests_paired_pixels_with_weights.csv'
    pd.DataFrame({
        'ndvi_before': ndvi_before_paired,
        'ndvi_after': ndvi_after_paired,
        'ndvi_change_after_minus_before': ndvi_delta_paired,
        'forest_weight': weights_paired,
    }).to_csv(paired_csv, index=False)
    print('Saved:', paired_csv)
else:
    print('CSV export skipped (SAVE_CSVS=False)')


In [ ]:
# Substantial change analysis for forests (relative and absolute thresholds)
# Mirrors mangrove substantial-change logic, with weighted and unweighted outputs.

REL_THRESHOLD = 0.10          # 10% relative change threshold
REL_BASELINE_MIN = 0.20       # compute relative % only where baseline NDVI >= this value
ABS_DELTA_THRESHOLD = 0.05    # absolute NDVI-unit threshold (noise-robust comparator)

# Relative change where baseline is high enough to avoid unstable percentages
eligible_rel = ndvi_before_paired >= REL_BASELINE_MIN
rel_change = np.full_like(ndvi_delta_paired, np.nan, dtype='float32')
rel_change[eligible_rel] = ndvi_delta_paired[eligible_rel] / ndvi_before_paired[eligible_rel]

substantial_rel = np.abs(rel_change) > REL_THRESHOLD
substantial_abs = np.abs(ndvi_delta_paired) > ABS_DELTA_THRESHOLD

w = weights_paired
w_rel = w[eligible_rel]

summary_substantial_forests = pd.DataFrame([
    {
        'definition': f'Relative: abs((after-before)/before) > {REL_THRESHOLD:.0%}',
        'baseline_condition': f'before >= {REL_BASELINE_MIN:.2f}',
        'n_eligible': int(np.sum(eligible_rel)),
        'n_substantial': int(np.nansum(substantial_rel)),
        'pct_substantial_of_eligible_unweighted': float(100.0 * np.nansum(substantial_rel) / max(np.sum(eligible_rel), 1)),
        'pct_substantial_of_eligible_weighted': float(100.0 * np.nansum(w_rel[substantial_rel[eligible_rel]]) / max(np.sum(w_rel), 1e-12)),
        'pct_substantial_decline_unweighted': float(100.0 * np.nansum(rel_change < -REL_THRESHOLD) / max(np.sum(eligible_rel), 1)),
        'pct_substantial_decline_weighted': float(100.0 * np.nansum(w_rel[rel_change[eligible_rel] < -REL_THRESHOLD]) / max(np.sum(w_rel), 1e-12)),
        'pct_substantial_improve_unweighted': float(100.0 * np.nansum(rel_change > REL_THRESHOLD) / max(np.sum(eligible_rel), 1)),
        'pct_substantial_improve_weighted': float(100.0 * np.nansum(w_rel[rel_change[eligible_rel] > REL_THRESHOLD]) / max(np.sum(w_rel), 1e-12)),
    },
    {
        'definition': f'Absolute: abs(after-before) > {ABS_DELTA_THRESHOLD:.2f}',
        'baseline_condition': 'none',
        'n_eligible': int(ndvi_delta_paired.size),
        'n_substantial': int(np.sum(substantial_abs)),
        'pct_substantial_of_eligible_unweighted': float(100.0 * np.sum(substantial_abs) / ndvi_delta_paired.size),
        'pct_substantial_of_eligible_weighted': float(100.0 * np.sum(w[substantial_abs]) / max(np.sum(w), 1e-12)),
        'pct_substantial_decline_unweighted': float(100.0 * np.sum(ndvi_delta_paired < -ABS_DELTA_THRESHOLD) / ndvi_delta_paired.size),
        'pct_substantial_decline_weighted': float(100.0 * np.sum(w[ndvi_delta_paired < -ABS_DELTA_THRESHOLD]) / max(np.sum(w), 1e-12)),
        'pct_substantial_improve_unweighted': float(100.0 * np.sum(ndvi_delta_paired > ABS_DELTA_THRESHOLD) / ndvi_delta_paired.size),
        'pct_substantial_improve_weighted': float(100.0 * np.sum(w[ndvi_delta_paired > ABS_DELTA_THRESHOLD]) / max(np.sum(w), 1e-12)),
    }
]).round(3)

summary_substantial_forests


In [ ]:
# Geospatial map: substantial relative NDVI change in forest-equivalent areas
# Red = NDVI decreased by >10%; Green = NDVI increased by >10%.

# Full-grid relative change only where paired-valid and baseline is high enough
eligible_rel_full = valid_paired_forest & (ndvi_before >= REL_BASELINE_MIN)
rel_change_full = np.full(ndvi_before.shape, np.nan, dtype='float32')
rel_change_full[eligible_rel_full] = (ndvi_after[eligible_rel_full] - ndvi_before[eligible_rel_full]) / ndvi_before[eligible_rel_full]

# Classify: -1 substantial decline, 0 other, +1 substantial improvement
cls_rel = np.full(ndvi_before.shape, np.nan, dtype='float32')
cls_rel[valid_paired_forest] = 0.0
cls_rel[eligible_rel_full & (rel_change_full < -REL_THRESHOLD)] = -1.0
cls_rel[eligible_rel_full & (rel_change_full > REL_THRESHOLD)] = 1.0

n_dec = int(np.nansum(cls_rel == -1.0))
n_mid = int(np.nansum(cls_rel == 0.0))
n_inc = int(np.nansum(cls_rel == 1.0))
n_tot = n_dec + n_mid + n_inc

cmap = ListedColormap(['#d73027', '#d9d9d9', '#1a9850'])
cmap.set_bad(color='white', alpha=1.0)
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)

fig, ax = plt.subplots(figsize=(12.5, 9.5), constrained_layout=True)
ax.imshow(
    cls_rel,
    cmap=cmap,
    norm=norm,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    origin='upper',
    interpolation='nearest',
)

ax.set_title(
    f'Forest-equivalent Areas: Relative NDVI Substantial Change (>10%)\nBaseline condition: NDVI before >= {REL_BASELINE_MIN:.2f}',
    fontsize=12,
)
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

legend_handles = [
    mpatches.Patch(facecolor='#d73027', edgecolor='none', label='NDVI decrease > 10%'),
    mpatches.Patch(facecolor='#d9d9d9', edgecolor='none', label='Not substantial / low baseline'),
    mpatches.Patch(facecolor='#1a9850', edgecolor='none', label='NDVI increase > 10%'),
]
ax.legend(
    handles=legend_handles,
    loc='upper right',
    frameon=True,
    framealpha=0.92,
    fontsize=9,
    handlelength=0.9,
    handleheight=0.9,
    borderpad=0.35,
    labelspacing=0.3,
)

ax.text(
    0.01,
    0.01,
    f'n={n_tot:,}  dec={100*n_dec/max(n_tot,1):.1f}%  other={100*n_mid/max(n_tot,1):.1f}%  inc={100*n_inc/max(n_tot,1):.1f}%',
    transform=ax.transAxes,
    fontsize=9,
    ha='left',
    va='bottom',
    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85, edgecolor='none'),
)

rel_map_png = output_dir / 'ndvi_forests_relative_substantial_change_gt10_map.png'
if SAVE_PNGS:
    fig.savefig(rel_map_png, dpi=320)
    print('Saved:', rel_map_png)
else:
    print('PNG export skipped (SAVE_PNGS=False):', rel_map_png)
plt.show()


In [ ]:
# Optional export: substantial-change summary for forests
if SAVE_CSVS:
    substantial_csv = output_dir / 'ndvi_forests_substantial_change_summary.csv'
    substantial_map_counts_csv = output_dir / 'ndvi_forests_relative_substantial_change_gt10_map_counts.csv'

    summary_substantial_forests.to_csv(substantial_csv, index=False)
    pd.DataFrame([
        {
            'n_total_valid_paired_forest': int(n_tot),
            'n_decrease_gt_10pct': int(n_dec),
            'n_increase_gt_10pct': int(n_inc),
            'n_other_not_substantial_or_low_baseline': int(n_mid),
            'pct_decrease_gt_10pct': float(100.0 * n_dec / max(n_tot, 1)),
            'pct_increase_gt_10pct': float(100.0 * n_inc / max(n_tot, 1)),
            'pct_other': float(100.0 * n_mid / max(n_tot, 1)),
            'relative_threshold': REL_THRESHOLD,
            'relative_baseline_min': REL_BASELINE_MIN,
        }
    ]).to_csv(substantial_map_counts_csv, index=False)

    print('Saved:', substantial_csv)
    print('Saved:', substantial_map_counts_csv)
else:
    print('CSV export skipped (SAVE_CSVS=False)')


## Notes
- Mixed classes are included using `forest_flood_equivalent_classes` fractions from `mixed_land_use_fractions`.
- Weighted summaries/histograms better reflect partial-forest classes than unweighted counts.
- `same` is exact zero change; small changes are counted as up/down.


## Land-Use Classes Within Forest Pixels That Increased NDVI

This section breaks down *which 2013 land-use classes* (from `Classify`) contributed forest-equivalent area in pixels where NDVI increased.

- `increase` means `after - before > 0` in paired-valid forest pixels.
- Mixed classes are weighted by their forest-equivalent fraction from `mixed_land_use_fractions`.



In [ ]:
# Forest NDVI increase by land-use class (weighted by forest-equivalent fraction)
pixel_area_m2 = abs(float(raster_transform.a * raster_transform.e))

ndvi_delta_full = ndvi_after - ndvi_before
increase_mask = valid_paired_forest & (ndvi_delta_full > 0.0)

# Relative substantial increase mask for optional comparison (>10%, baseline >= 0.20)
eligible_rel_full_local = valid_paired_forest & (ndvi_before >= REL_BASELINE_MIN)
rel_change_full_local = np.full(ndvi_before.shape, np.nan, dtype='float32')
rel_change_full_local[eligible_rel_full_local] = (
    ndvi_after[eligible_rel_full_local] - ndvi_before[eligible_rel_full_local]
) / ndvi_before[eligible_rel_full_local]
substantial_increase_mask = eligible_rel_full_local & (rel_change_full_local > REL_THRESHOLD)

rows = []
for class_name, grp in forest_landcover.groupby('Classify', dropna=False):
    shapes_class = ((geom, float(frac)) for geom, frac in zip(grp.geometry, grp['forest_fraction']))
    class_frac_raster = rasterize(
        shapes=shapes_class,
        out_shape=raster_shape,
        transform=raster_transform,
        fill=0.0,
        dtype='float32',
    )

    total_weighted_px = float(class_frac_raster[valid_paired_forest].sum())
    inc_weighted_px = float(class_frac_raster[increase_mask].sum())
    sub_inc_weighted_px = float(class_frac_raster[substantial_increase_mask].sum())

    rows.append({
        'Classify': str(class_name),
        'forest_fraction_used': float(grp['forest_fraction'].iloc[0]),
        'paired_forest_equiv_area_ha': total_weighted_px * pixel_area_m2 / 10_000.0,
        'increased_ndvi_forest_equiv_area_ha': inc_weighted_px * pixel_area_m2 / 10_000.0,
        'substantial_increase_gt10pct_area_ha': sub_inc_weighted_px * pixel_area_m2 / 10_000.0,
    })

forest_ndvi_increase_by_landuse = pd.DataFrame(rows)
forest_ndvi_increase_by_landuse['pct_of_class_area_with_increase'] = np.where(
    forest_ndvi_increase_by_landuse['paired_forest_equiv_area_ha'] > 0,
    100.0 * forest_ndvi_increase_by_landuse['increased_ndvi_forest_equiv_area_ha'] / forest_ndvi_increase_by_landuse['paired_forest_equiv_area_ha'],
    np.nan,
)

total_increase_ha = forest_ndvi_increase_by_landuse['increased_ndvi_forest_equiv_area_ha'].sum()
forest_ndvi_increase_by_landuse['pct_of_all_increase_area'] = np.where(
    total_increase_ha > 0,
    100.0 * forest_ndvi_increase_by_landuse['increased_ndvi_forest_equiv_area_ha'] / total_increase_ha,
    np.nan,
)

forest_ndvi_increase_by_landuse = forest_ndvi_increase_by_landuse.sort_values(
    'increased_ndvi_forest_equiv_area_ha', ascending=False
).reset_index(drop=True)

display(forest_ndvi_increase_by_landuse.round(3))

if SAVE_CSVS:
    out_csv = output_dir / 'ndvi_forests_increase_by_landuse_class_weighted.csv'
    forest_ndvi_increase_by_landuse.to_csv(out_csv, index=False)
    print('Saved:', out_csv)
else:
    print('CSV export skipped (SAVE_CSVS=False)')



## Map: Forest NDVI increase >10% classes only

This map shows only forest pixels with substantial relative NDVI increase (`(after - before) / before > 10%`).
Baseline condition follows the notebook threshold: `NDVI before >= 0.20`.
Each pixel is coloured by the dominant intersecting 2013 land-use class (weighted by forest fraction for mixed classes).


In [ ]:
# Map only substantial NDVI-increase forest pixels (>10% relative), coloured by dominant land-use class
eligible_rel_full_map = valid_paired_forest & (ndvi_before >= REL_BASELINE_MIN)
rel_change_full_map = np.full(ndvi_before.shape, np.nan, dtype='float32')
rel_change_full_map[eligible_rel_full_map] = (
    ndvi_after[eligible_rel_full_map] - ndvi_before[eligible_rel_full_map]
) / ndvi_before[eligible_rel_full_map]
increase_mask = eligible_rel_full_map & (rel_change_full_map > REL_THRESHOLD)

class_area = forest_ndvi_increase_by_landuse.copy()
class_area = class_area[class_area['substantial_increase_gt10pct_area_ha'] > 0].copy()
class_names = class_area['Classify'].tolist()

if len(class_names) == 0:
    print('No forest pixels found with NDVI increase >10%.')
else:
    stack = np.zeros((len(class_names), raster_shape[0], raster_shape[1]), dtype='float32')

    for i, class_name in enumerate(class_names):
        grp = forest_landcover[forest_landcover['Classify'] == class_name]
        shapes_class = ((geom, float(frac)) for geom, frac in zip(grp.geometry, grp['forest_fraction']))
        class_frac_raster = rasterize(
            shapes=shapes_class,
            out_shape=raster_shape,
            transform=raster_transform,
            fill=0.0,
            dtype='float32',
        )
        stack[i] = class_frac_raster

    stack[:, ~increase_mask] = 0.0
    max_vals = stack.max(axis=0)
    argmax_idx = stack.argmax(axis=0)

    dominant_class = np.zeros(raster_shape, dtype='int16')
    valid_dom = increase_mask & (max_vals > 0)
    dominant_class[valid_dom] = argmax_idx[valid_dom] + 1

    # Discrete map colours: 0 = transparent background; 1..N = classes
    class_colors = plt.cm.tab20(np.linspace(0, 1, len(class_names)))
    rgba0 = np.array([[1.0, 1.0, 1.0, 0.0]])
    cmap = ListedColormap(np.vstack([rgba0, class_colors]))
    norm = BoundaryNorm(np.arange(-0.5, len(class_names) + 1.5, 1), cmap.N)

    fig, ax = plt.subplots(figsize=(13.5, 9.5), constrained_layout=True)
    ax.imshow(
        dominant_class,
        cmap=cmap,
        norm=norm,
        extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
        origin='upper',
        interpolation='nearest',
    )

    ax.set_title(
        f'Forest Pixels with NDVI Increase >10%: Dominant Intersecting Land-Use Class\n'
        f'Baseline condition: NDVI before >= {REL_BASELINE_MIN:.2f}',
        fontsize=12,
    )
    ax.set_xlabel('Easting (m, EPSG:3448)')
    ax.set_ylabel('Northing (m, EPSG:3448)')
    ax.set_aspect('equal')

    handles = []
    for i, row in class_area.reset_index(drop=True).iterrows():
        label = f"{row['Classify']} ({row['substantial_increase_gt10pct_area_ha']:.0f} ha)"
        handles.append(mpatches.Patch(facecolor=class_colors[i], edgecolor='none', label=label))

    ax.legend(
        handles=handles,
        title='NDVI increase >10% classes',
        loc='upper left',
        bbox_to_anchor=(1.02, 1.0),
        frameon=True,
        fontsize=8,
        title_fontsize=9,
        borderpad=0.3,
        labelspacing=0.25,
    )

    increase_class_map_png = output_dir / 'ndvi_forests_increase_gt10_classes_map.png'
    if SAVE_PNGS:
        fig.savefig(increase_class_map_png, dpi=320, bbox_inches='tight')
        print('Saved:', increase_class_map_png)
    else:
        print('PNG export skipped (SAVE_PNGS=False):', increase_class_map_png)

    plt.show()

